# 🤖 Building an ASL Alphabet Classifier

Train a CNN to recognise ASL fingerspelling letters from images.

**Steps:**
1. Load dataset
2. Train/val/test split
3. Define CNN model
4. Train & evaluate
5. Confusion matrix
6. Save & load model

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Dataset

Using ASL-MNIST (CSV) for simplicity. Download first:
```bash
kaggle datasets download -d datamunge/sign-language-mnist
unzip sign-language-mnist.zip -d data/asl_mnist/
```

In [ ]:
from scripts.data_loader import ASLMNISTDataset

train_csv = '../data/asl_mnist/sign_mnist_train.csv'
test_csv  = '../data/asl_mnist/sign_mnist_test.csv'

full_train = ASLMNISTDataset(train_csv)
test_ds   = ASLMNISTDataset(test_csv)

print(f'Full train: {len(full_train)}, Test: {len(test_ds)}, Classes: {full_train.num_classes}')

## 2. Train / Val Split

In [ ]:
val_size = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size], generator=torch.Generator(42))

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

## 3. Define CNN Model

In [ ]:
class ASLCNN(nn.Module):
    """Simple CNN for 28x28 grayscale images."""

    def __init__(self, num_classes: int = 24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = ASLCNN(num_classes=full_train.num_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)

epochs = 15
train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, epochs + 1):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    train_loss = running_loss / len(train_ds)
    train_losses.append(train_loss)

    # --- Validate ---
    model.eval()
    correct, total, vloss = 0, 0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch['image'].to(device), batch['label'].to(device)
            logits = model(images)
            vloss += criterion(logits, labels).item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_loss = vloss / len(val_ds)
    val_acc = correct / total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    scheduler.step(val_loss)
    print(f'Epoch {epoch:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train')
ax1.plot(val_losses, label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(val_accs, label='Val Accuracy', color='green')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 5. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', annot=False, fmt='d')
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()

## 6. Save & Load Model

In [ ]:
# Save
torch.save({
    'model_state': model.state_dict(),
    'num_classes': full_train.num_classes,
}, '../models/asl_cnn.pth')
print('Model saved to models/asl_cnn.pth')

# Load
ckpt = torch.load('../models/asl_cnn.pth', map_location=device)
loaded_model = ASLCNN(num_classes=ckpt['num_classes']).to(device)
loaded_model.load_state_dict(ckpt['model_state'])
print('Model loaded successfully')